# 09 — Bundle adjustment, named-axis style

*A small bundle-adjustment (BA) problem expressed end-to-end with `_tex` literals and named axes.*

Goals of this notebook:

1. Frame BA --- the workhorse refinement step of Structure-from-Motion --- as a problem whose **axis structure is what makes it hard to write correctly** in positional tensor libraries.
2. Express the reprojection residual as a single `_tex` LaTeX-math literal whose subscripts (`view`, `landmark`) match the named axes of the bound tensors.
3. Watch a tiny synthetic scene converge under repeated camera updates, all with no nested `for` loops over axes.

**Scope:** a one-dimensional toy version that fits in one notebook. The full three-dimensional pinhole-camera version, evaluated on the ETH3D *courtyard* and Strecha *fountain-P11* scenes, is the case study of the project's MLSys 2027 paper (`paper/mlsys-2027/`, §6.2); the harness there reuses the same formula structure shown below.

**Prerequisites:** `01_formula-is-the-program.ipynb` for the `_tex` UDL mechanics; `05_autograd-from-scratch.ipynb` if you want to extend this to a real Gauss--Newton solver.

## 1 — Why BA is a good test of named-axis discipline

A bundle-adjustment problem has four natural axes:

| Axis | Meaning | Typical extent |
| ---- | ------- | -------------- |
| `view` | camera index | tens to thousands |
| `landmark` | 3D point index | hundreds to millions |
| `pose-dof` | degrees of freedom per camera (rotation + translation) | 6 |
| `obs` | image-plane coordinate (`u`, `v`) | 2 |

The classic bug at this scale is **transposing `view` and `landmark`** in a Jacobian assembly: the shapes still line up (`(V, L, 2)` vs `(L, V, 2)` are both rank-3) and the optimisation **runs**, but it converges to a different minimum, slowly, with no error message.

A named-axis system refuses this transposition because the names do not match. The library carries the names through every contraction; you write the math, the type system enforces the geometry.

## 2 — Setup

In [ ]:
#pragma cling add_include_path("../include")
#include <cmath>
#include <iomanip>
#include <iostream>
#include <random>
#include <tensor/core/core.hpp>
#include <tensor/tex/tex.hpp>

using tensor::core::Axis;
using tensor::core::DynamicShape;
using tensor::core::DynamicTensor;
using tensor::tex::Evaluator;
using namespace tensor::tex::literals;   // brings in operator""_tex

## 3 — A 1-D miniature scene

We strip BA down to its named-axis essentials with a one-dimensional world:

- `landmark`s are scalar positions $p_i \in \mathbb{R}$ along a line (known ground truth).
- `view`s are scalar camera positions $c_j \in \mathbb{R}$ along the same line (the unknowns we want to recover).
- The observation model is the relative position the camera measures:

$$
\text{obs}_{ij} = p_i - c_j + \epsilon_{ij},\qquad \epsilon_{ij}\sim\mathcal{N}(0, \sigma^2).
$$

This is degenerate as real BA (no rotation, no perspective division) but it has the right *axis shape* --- a (`landmark`, `view`) residual tensor formed from a `landmark`-rank tensor and a `view`-rank tensor --- which is the part we want to type-check.

In [ ]:
constexpr std::size_t L = 5;    // number of landmarks
constexpr std::size_t V = 3;    // number of cameras

// Ground-truth landmarks (known) and cameras (unknown — we will recover these).
DynamicTensor<double> p(DynamicShape{Axis{"landmark", L}}, {1.0, 2.5, 4.0, 6.0, 8.0});
DynamicTensor<double> c_true(DynamicShape{Axis{"view", V}}, {0.0, 1.5, 3.0});

// Synthesise noisy observations: obs[i,j] = p[i] - c_true[j] + noise.
std::mt19937 rng(42);
std::normal_distribution<double> noise(0.0, 0.05);

std::vector<double> obs_data(L * V);
for (std::size_t i = 0; i < L; ++i)
    for (std::size_t j = 0; j < V; ++j)
        obs_data[i*V + j] = p[i] - c_true[j] + noise(rng);

DynamicTensor<double> obs(DynamicShape{Axis{"landmark", L}, Axis{"view", V}},
                          std::move(obs_data));

std::cout << "True camera positions: "; for (std::size_t j = 0; j < V; ++j) std::cout << c_true[j] << ' '; std::cout << std::endl;
std::cout << "obs.shape() rank=" << obs.shape().rank() << ", axes={landmark, view}" << std::endl;

## 4 — The reprojection residual as a `_tex` literal

The residual we want to compute is

$$
r_{ij} = p_i - c_j - \text{obs}_{ij}.
$$

In the library, that statement is **the program**: we write the formula as a LaTeX-math literal, bind the names `p`, `c`, `obs` to the tensors we built above, and the Evaluator produces $r$ with the correct `(landmark, view)` axis structure.

Note what does *not* appear in the code below: no nested `for` loops, no explicit broadcasting calls, no positional axis indices.

In [ ]:
// Initial guess: all cameras at 0.
DynamicTensor<double> c(DynamicShape{Axis{"view", V}}, {0.0, 0.0, 0.0});

Evaluator<double> ev;
ev.bind("p",   p);
ev.bind("c",   c);
ev.bind("obs", obs);

// The formula and the program are the same string.
auto residual = ev.evaluate(R"(r_{ij} = p_i - c_j - obs_{ij})"_tex);

std::cout << "r.shape() rank=" << residual.shape().rank()
          << " (expected 2: landmark × view)" << std::endl;
std::cout << "First few residuals: ";
for (std::size_t k = 0; k < std::min<std::size_t>(6, residual.size()); ++k)
    std::cout << std::setprecision(3) << residual[k] << ' ';
std::cout << std::endl;

## 5 — Sum-of-squares loss

The least-squares cost is

$$
L = \frac{1}{2}\sum_{i,j} r_{ij}^{\,2}.
$$

The same `_tex` mechanism expresses the inner sum. We could either bind `r` back to the Evaluator and write the squared sum as another literal, or compute it directly from the residual tensor; we do the latter here for clarity.

In [ ]:
auto loss_of = [](DynamicTensor<double> const& r) {
    double s = 0.0;
    for (std::size_t k = 0; k < r.size(); ++k) s += r[k] * r[k];
    return 0.5 * s;
};

std::cout << "Initial loss = " << std::setprecision(4) << loss_of(residual) << std::endl;
std::cout << "(Should be large — cameras are all at 0, true positions are 0, 1.5, 3.)" << std::endl;

## 6 — Iterative refinement

For this scalar problem, the closed-form optimum of each $c_j$ given the others is

$$
c_j^\ast = \frac{1}{L}\sum_i\bigl(p_i - \text{obs}_{ij}\bigr).
$$

We take a step in this direction at each iteration and watch the loss decay. (For the three-dimensional pinhole-camera version evaluated in the MLSys paper, the update is Gauss--Newton on the autograd-derived Jacobian; see `tutorials/05_autograd-from-scratch.ipynb` for the building blocks and the paper's `bench/` harness for the full implementation.)

In [ ]:
constexpr std::size_t MAX_ITERS = 8;
constexpr double STEP = 0.5;

std::cout << "iter | loss      | c_0     c_1     c_2\n";
std::cout << "-----+-----------+---------------------\n";
for (std::size_t it = 0; it < MAX_ITERS; ++it) {
    // Recompute the residual with the current c.
    ev.bind("c", c);
    auto r = ev.evaluate(R"(r_{ij} = p_i - c_j - obs_{ij})"_tex);

    // Print progress.
    std::cout << std::setw(4) << it << " | "
              << std::fixed << std::setprecision(5) << std::setw(9) << loss_of(r)
              << " |";
    for (std::size_t j = 0; j < V; ++j)
        std::cout << ' ' << std::setw(7) << c[j];
    std::cout << std::endl;

    // Update each c_j by stepping toward its coordinate-descent optimum.
    std::vector<double> c_new(V);
    for (std::size_t j = 0; j < V; ++j) {
        double acc = 0.0;
        for (std::size_t i = 0; i < L; ++i)
            acc += (p[i] - obs[i*V + j]);
        double c_star = acc / static_cast<double>(L);
        c_new[j] = (1.0 - STEP) * c[j] + STEP * c_star;
    }
    c = DynamicTensor<double>(DynamicShape{Axis{"view", V}}, std::move(c_new));
}

std::cout << "\nGround truth: "; for (std::size_t j = 0; j < V; ++j) std::cout << c_true[j] << ' '; std::cout << std::endl;
std::cout << "Recovered:    "; for (std::size_t j = 0; j < V; ++j) std::cout << c[j]      << ' '; std::cout << std::endl;

## 7 — What changes when you scale this up

Everything above generalises to real bundle adjustment essentially by swapping in richer geometry; the *named-axis structure does not change*:

| 1-D toy (this notebook) | 3-D pinhole BA (paper §6.2) |
| ------------------------ | ---------------------------- |
| `p` is rank-1 `(landmark,)` | `p` is rank-2 `(landmark, xyz)` |
| `c` is rank-1 `(view,)` | `c` is rank-2 `(view, pose-dof)` (6 DoF: rotation + translation) |
| `obs` is rank-2 `(landmark, view)` | `obs` is rank-3 `(landmark, view, uv)` |
| Residual is rank-2 `(landmark, view)` | Residual is rank-3 `(landmark, view, uv)` |
| Formula is `p_i - c_j - obs_{ij}` | Formula is the pinhole reprojection in `_tex`: `\sum_k R_{jk} (p_{ik} - c_{jk}) / \ldots` |
| Closed-form coordinate update | Gauss--Newton with autograd-derived Jacobian |
| 3 cameras × 5 landmarks | 50 views × 200 landmarks (ETH3D *courtyard*) |

The bench harness `bench/bench_mlsys_paper_case_study.py` consumes ETH3D and Strecha datasets, builds the residual tensors with the same axis names used here, and evaluates the same formula across the three substrates of the Hexagonal-lite kernel-backend port (`reference`, `Eigen`, WebGPU-via-Dawn). The named-axis discipline that made *this* notebook readable is the same discipline that makes the paper's case study scale.

### Further reading

- `01_formula-is-the-program.ipynb` --- the `_tex` UDL by itself, before BA enters the picture.
- `05_autograd-from-scratch.ipynb` --- the autograd primitives needed for the real Gauss--Newton step.
- `08_swappable-backends.ipynb` --- how the `reference` / `Eigen` / WebGPU backends are selected at CMake configure time.
- `paper/mlsys-2027/main.tex` §6.2 --- the case-study results table.